In [9]:
!pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------ --------------------- 0.8/1.7 MB 6.2 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 3.3 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
%%writefile app.py
import os
import math
import textwrap
from typing import List

import streamlit as st
from serpapi.google_search import GoogleSearch


# Transformers imports are deferred until needed to avoid long import times during UI load

# ----------------------- Helpers -----------------------
def chunk_text(text: str, max_chars: int = 12000) -> List[str]:
    """Naive chunker that slices text into pieces <= max_chars at sentence boundaries."""
    if len(text) <= max_chars:
        return [text]
    sentences = text.replace("\n", " ").split('. ')
    chunks = []
    cur = []
    cur_len = 0
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        add_len = len(s) + 2  # approximate
        if cur_len + add_len > max_chars:
            chunks.append('. '.join(cur).strip() + '.')
            cur = [s]
            cur_len = len(s)
        else:
            cur.append(s)
            cur_len += add_len
    if cur:
        chunks.append('. '.join(cur).strip() + '.')
    return chunks

# ----------------------- Caching -----------------------
@st.cache_data(show_spinner=False)
def fetch_google_news(query: str, api_key: str, num_results: int = 10):
    """Fetch news using SerpAPI (google_news engine). Returns list of dicts."""
    if not api_key:
        return []
    # --- CORRECTION APPLIED HERE (4 SPACES OF INDENTATION ADDED) ---
    params = {
        "engine": "google_news",
        "q": query,
        "tbm": "nws",    # <-- REQUIRED FOR NEWS
        "api_key": api_key,
        "num": num_results,
    }
    # -----------------------------------------------------------------

    try:
        search = GoogleSearch(params)
        results = search.get_dict()
    except Exception as e:
        st.error(f"SerpAPI request failed: {e}")
        return []

    items = results.get("news_results") or results.get("news") or []
    articles = []
    for it in items:
        articles.append({
            "title": it.get("title"),
            "source": it.get("source"),
            "snippet": it.get("snippet") or it.get("summary") or "",
            "link": it.get("link"),
            "date": it.get("date") or it.get("published") or "",
        })
    return articles

@st.cache_resource(show_spinner=False)
def load_hf_summarizer(model_name: str = "sshleifer/distilbart-cnn-12-6"):
    """Load HF summarization pipeline in a safe CPU mode to avoid meta tensor issues.

    Returns a callable summarizer(text) -> summary_text
    """
    from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

    # Load tokenizer and model explicitly with no device map (avoid meta tensors)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map=None)

    # Force materialization onto CPU (avoid meta tensors)
    model.to("cpu")
    try:
        model.tie_weights()
    except Exception:
        pass

    summarizer = pipeline(
        "summarization",
        model=model,
        tokenizer=tokenizer,
        device=-1,  # CPU
    )

    def _summarize(text: str, max_length: int = 200, min_length: int = 30):
        chunks = chunk_text(text, max_chars=8000)
        out_chunks = []
        for c in chunks:
            try:
                r = summarizer(c, max_length=max_length, min_length=min_length, do_sample=False)
                out_chunks.append(r[0]["summary_text"])
            except Exception as e:
                st.warning(f"Local summarizer failed on a chunk: {e}")
        return "\n\n".join(out_chunks)

    return _summarize

# ----------------------- Streamlit UI -----------------------
st.set_page_config(page_title="AI News Orchestrator", layout="wide")
st.title("AI News Orchestrator")

with st.sidebar:
    st.header("Configuration")
    serpapi_key = st.text_input("SerpAPI API Key", type="password", placeholder="sk-...", help="Get from https://serpapi.com/")
    summarizer_option = st.radio("Summarizer", ["HuggingFace Local (CPU)", "OpenAI API (chat)"])
    if summarizer_option == "OpenAI API (chat)":
        openai_key = st.text_input("OpenAI API Key", type="password", placeholder="sk-...")
    else:
        openai_key = None

    st.markdown("---")
    st.write("Tips:")
    st.write("• SerpAPI is the easiest way to query Google News. Free tier exists but limited.")
    st.write("• Local HuggingFace summarizer runs on CPU; pick a small model for speed.")

query = st.text_input("Search topic", value="OpenAI")
num = st.slider("Number of articles", 1, 50, 10)
max_len = st.slider("Summary max length (tokens approx)", 50, 512, 200)
min_len = st.slider("Summary min length (tokens approx)", 10, 200, 30)

col1, col2 = st.columns([3, 1])
with col2:
    if st.button("Fetch & Summarize"):
        if not serpapi_key:
            st.error("Provide SerpAPI key in the sidebar.")
        else:
            with st.spinner("Fetching articles..."):
                articles = fetch_google_news(query, serpapi_key, num_results=num)

            if not articles:
                st.warning("No articles returned. Check your API key or query.")
            else:
                st.success(f"Fetched {len(articles)} articles")

                combined = "\n\n".join([
                    f"{a['title']} — {a['source']}\n{a['snippet']}"
                    for a in articles
                ])

                if summarizer_option == "HuggingFace Local (CPU)":
                    with st.spinner("Loading local summarizer (this may take a few seconds)..."):
                        try:
                            hf_summarize = load_hf_summarizer()
                        except Exception as e:
                            st.error(f"Failed to load local summarizer: {e}")
                            hf_summarize = None

                    if hf_summarize:
                        with st.spinner("Summarizing (local)..."):
                            summary = hf_summarize(combined, max_length=max_len, min_length=min_len)

                else:
                    if not openai_key:
                        st.error("Provide OpenAI API key in the sidebar to use OpenAI summarizer.")
                        summary = ""
                    else:
                        try:
                            import openai
                            openai.api_key = openai_key
                            prompt = (
                                "Summarize the following news articles into a concise overview (3-6 bullet points). "
                                "Include main events, actors, and dates.\n\n" + combined
                            )
                            resp = openai.ChatCompletion.create(
                                model="gpt-4o-mini",
                                messages=[{"role": "user", "content": prompt}],
                                max_tokens=500,
                                temperature=0.0,
                            )
                            summary = resp["choices"][0]["message"]["content"].strip()
                        except Exception as e:
                            st.error(f"OpenAI summarization failed: {e}")
                            summary = ""

                st.markdown("## Summary")
                st.write(summary)

                st.markdown("## Articles")
                for a in articles:
                    st.markdown(f"**{a['title']}** ")
                    st.write(f"{a['source']} — {a['date']}")
                    st.write(a['snippet'])
                    st.write(a['link'])
                    st.markdown("---")

# ----------------------- Footer / Requirements -----------------------
st.sidebar.markdown("---")
st.sidebar.write("Requirements:")
st.sidebar.code('\n'.join([
    "streamlit",
    "serpapi",
    "transformers>=4.0",
    "torch",
    "openai  # optional if you want OpenAI summarizer"
]))

st.caption("App built to fetch Google News through SerpAPI and summarize via HuggingFace or OpenAI.")

Overwriting app.py


In [ ]:
!streamlit run app.py